# Run & verify

**Is the deployment sound, can I run a scan, and are the tables consistent?**

The only notebook here that writes anything. The other six read what this one produced.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose.
Set the notebook to **Run accessed commands** (the dropdown beside *Run all*) if you want a
widget change to re-run the cells that depend on it — otherwise you will change the filter and
read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Charts that span scans say so in their title.

In [ ]:
PAGE = {"disappearance": ("scan_ts", ["scan_ts", "midpoint"])}

import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles
import run_pipeline
import pandas as pd
from pyspark.sql import functions as F

# --- Load workspace CSV files as Spark temp views ---
_csv_dir = "/Workspace/Users/nicolai.drobyshevski@decathlon.com/wiz/csv_export"
_scope = dbutils.widgets.get("scope") if dbutils.widgets.get("scope") else "os"
_prefix = f"wiz_{_scope}_"

print(f"Loading CSVs from: {_csv_dir}")
print(f"Scope: {_scope}\n")

# Load scans
_path = os.path.join(_csv_dir, f"{_prefix}scans.csv")
_pdf_scans = pd.read_csv(_path)
_pdf_scans["scope"] = _scope
spark.createDataFrame(_pdf_scans).createOrReplaceTempView(f"{_prefix}scans")
print(f"  ✓ {_prefix}scans ({len(_pdf_scans)} rows)")

# Load ledger
_path = os.path.join(_csv_dir, f"{_prefix}vuln_ledger.csv")
_pdf_ledger = pd.read_csv(_path)
if "scope" not in _pdf_ledger.columns:
    _pdf_ledger["scope"] = _scope
spark.createDataFrame(_pdf_ledger).createOrReplaceTempView(f"{_prefix}vuln_ledger")
print(f"  ✓ {_prefix}vuln_ledger ({len(_pdf_ledger)} rows)")

# Show latest scan info
print(f"\n{'='*60}")
print("  LATEST SCAN (from CSV)")
print(f"{'='*60}")
_latest = _pdf_scans.sort_values('ts', ascending=False).iloc[0]
print(f"  scan_id: {_latest['scan_id']}")
print(f"  ts:      {_latest['ts']}")
print(f"  total:   {_latest['total']}")
print(f"  new:     {_latest['new_count']} | resolved: {_latest['resolved_count']}")

print(f"\n{'='*60}")
print("  LEDGER SUMMARY")
print(f"{'='*60}")
print(f"  Total vulnerabilities: {len(_pdf_ledger)}")
if 'severity' in _pdf_ledger.columns:
    print(f"  By severity:")
    for sev, count in _pdf_ledger['severity'].value_counts().items():
        print(f"    {sev:12} {count:>6}")
if 'status' in _pdf_ledger.columns:
    print(f"  By status:")
    for status, count in _pdf_ledger['status'].value_counts().items():
        print(f"    {status:12} {count:>6}")

print(f"\n⚠️  Note: Gold metrics tables (mttr, program, capacity) are not available as CSV.")
print(f"    The full panels.context() requires a pipeline run to compute them.")
print(f"    Run notebook 06 Cell 8 (with write access) to produce the gold tables.")

## Is the deployment sound?

Every module must report the same version **from the same folder**. A half-updated folder
imports cleanly and then fails much later at something that looks unrelated — the first v2
run hit exactly that, 137,870 findings in, as a schema mismatch that named neither the
stale file nor the fix.

The printed path is the other half: `config`, `metrics` and `ingest` are generic module
names, and so are `panels`, `figures` and `tiles`. If something else on `sys.path` shadows
one, the path below says so — an `AttributeError` three cells later would not.

In [ ]:
import config, dbx, ingest, ledger, metrics, run_pipeline

for _m in (config, dbx, ingest, ledger, metrics, run_pipeline, panels, figures, tiles):
    print(f"{_m.__name__:14} {getattr(_m, 'MODULE_VERSION', 'PRE-2.0 — STALE'):8}"
          f" {_m.__file__}")

## Run a scan

Needs credentials in a secret scope and write access to the schema. Set `catalog`,
`schema`, `wiz_api_url` and `secret_scope` in the widgets first.

Safe to re-run: the scan id is the idempotency guard, so a retry with the same id is a
no-op rather than a double count. `--rebuild_ledger` replays every archived scan and is
the one thing here that is not cheap.

**After editing or re-pasting any module, restart Python — do not reload.**
`dbutils.library.restartPython()`. `importlib.reload` is worse than doing nothing: it
re-executes a module while every other module still holds references into the old one.

In [ ]:
import importlib
import run_pipeline
# Reload to pick up the write_bronze_batch fix (as_path support)
run_pipeline = importlib.reload(run_pipeline)
from run_pipeline import main

# param() checks sys.argv FIRST, before widgets — this bypasses notebook query parameters
# that are stuck at "" and can't be overridden by dbutils.widgets.text
import sys
_data_path = "dbfs:/tmp/wiz_pipeline"
sys.argv = [sys.argv[0], f"--data_path={_data_path}"]

print(f"Pipeline will write Delta to: {_data_path}")
print(f"sys.argv override: {sys.argv}")
print("Running scan...\n")
result = main()
print(result)

# --- Export Delta tables to workspace CSV ---
import os
_csv_dir = "/Workspace/Users/nicolai.drobyshevski@decathlon.com/wiz/csv_export"
os.makedirs(_csv_dir, exist_ok=True)

_scope = run_pipeline.resolve_scope()
_tables = run_pipeline.resolve_tables("", _scope, data_path=_data_path)

print(f"\n{'='*60}")
print("  Exporting Delta tables to workspace CSV")
print(f"{'='*60}")

_export_map = [
    ("scans", _tables.scans),
    ("vuln_ledger", _tables.ledger),
    ("findings_raw", _tables.bronze),
    ("findings", _tables.silver),
    ("metrics_mttr", _tables.mttr),
    ("metrics_program", _tables.program),
    ("metrics_capacity", _tables.capacity),
    ("metrics_sensitivity", _tables.sensitivity),
]

for name, table_ref in _export_map:
    try:
        # Delta path is stored as: delta.`<path>`
        _delta_path = table_ref.replace("delta.`", "").rstrip("`")
        _df = spark.read.format("delta").load(_delta_path)
        _count = _df.count()
        _prefix = run_pipeline.default_table_prefix(_scope)
        _csv_path = os.path.join(_csv_dir, f"{_prefix}{name}.csv")
        _df.toPandas().to_csv(_csv_path, index=False)
        print(f"  \u2713 {name}: {_count} rows -> {_csv_path}")
    except Exception as e:
        print(f"  \u2717 {name}: {e}")

print(f"\nDone. CSVs in: {_csv_dir}")

## Are the tables consistent?

The three gold tables and the run log should agree on which scan is the latest. They
disagree when a run died between two writes; the pipeline refuses to start in that state,
and this surfaces it *before* the next run hits it — and before somebody reads a page
whose halves came from different scans.

The ledger is the exception by design: it is MERGEd current state and carries
`last_scan_id` rather than `scan_id`.

In [ ]:
display(panels.scan_pin_check(spark, ctx))

In [ ]:
display(panels.table_inventory(spark, ctx))

In [ ]:
display(panels.run_health(spark, ctx))

---

GAS's *Data* page also imports a legacy migration bundle. brick ingests from the Wiz API
and has no import path, so there is nothing to expose. To get data out, use the download
button on any result grid above.